# Laboratorio 2: motor de búsqueda semántica

Comparación entre embeddings multilingües y una búsqueda básica por palabras clave en un corpus de soporte para comercio electrónico.

[Link a Repositorio](https://github.com/donmatthiuz/NLP/tree/lab2)

In [1]:
%pip install -q sentence-transformers numpy scikit-learn

## Configuración y datos

In [2]:
import re
import unicodedata

import numpy as np
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TOP_K = 3

CORPUS = [
    "No puedo iniciar sesión en mi cuenta.",
    "Olvidé mi contraseña y necesito recuperar el acceso.",
    "La cuenta fue bloqueada por demasiados intentos fallidos.",
    "Puedes restablecer tu clave desde la página de acceso.",
    "La verificación en dos pasos no envía el código de seguridad.",
    "Quiero actualizar el correo electrónico asociado a mi perfil.",
    "No recibí el mensaje para confirmar mi correo.",
    "La aplicación móvil no carga correctamente.",
    "La aplicación se cierra sola después de abrirla.",
    "El sitio web está temporalmente fuera de servicio.",
    "Revisa tu conexión a internet antes de volver a intentarlo.",
    "Debes actualizar la aplicación a la versión más reciente.",
    "El pedido ya fue enviado y está en camino.",
    "Puedes rastrear el paquete con el número de seguimiento.",
    "La entrega de mi compra está retrasada.",
    "Necesito modificar la dirección de entrega de mi pedido.",
    "Quiero devolver un producto porque llegó dañado.",
    "El reembolso puede tardar cinco días hábiles.",
    "Mi tarjeta fue rechazada al intentar pagar.",
    "Me cobraron dos veces por la misma compra.",
    "La factura se puede descargar desde el historial de pedidos.",
    "Necesito comunicarme con un agente de soporte.",
    "Las notificaciones de nuevos pedidos están desactivadas.",
    "Deseo eliminar definitivamente mi perfil de usuario.",
]

CONSULTAS = [
    "La aplicación móvil no carga correctamente",
    "Perdí mis credenciales y necesito volver a entrar",
    "La cuenta no funciona",
    "Deseo cambiar el email vinculado a mi usuario",
    "¿Dónde está mi compra?",
    "El pago de mi pedido aparece duplicado",
]

## Generación de embeddings

In [3]:
modelo = SentenceTransformer(MODELO_EMBEDDINGS)
embeddings_corpus = modelo.encode(
    CORPUS,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print(f"Corpus: {len(CORPUS)} oraciones")
print(f"Dimensión de embeddings: {embeddings_corpus.shape[1]}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Corpus: 24 oraciones
Dimensión de embeddings: 384


## Búsqueda semántica y recuperación top-k

In [4]:
def similitud_coseno(embedding_consulta, embeddings_indexados):
    return embeddings_indexados @ embedding_consulta


def buscar_semanticamente(consulta, corpus, embeddings_indexados, modelo, top_k=3):
    if not consulta.strip():
        raise ValueError("La consulta no puede estar vacía.")
    if not 1 <= top_k <= len(corpus):
        raise ValueError(f"top_k debe estar entre 1 y {len(corpus)}.")

    embedding_consulta = modelo.encode(
        consulta,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    puntajes = similitud_coseno(embedding_consulta, embeddings_indexados)
    indices = np.argsort(puntajes)[::-1][:top_k]

    return [
        {
            "rank": posicion,
            "indice": int(indice),
            "texto": corpus[indice],
            "score": float(puntajes[indice]),
        }
        for posicion, indice in enumerate(indices, start=1)
    ]

## Búsqueda por palabras clave

In [5]:
def tokenizar_simple(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    return set(re.findall(r"\b[a-z0-9]+\b", texto))


def buscar_por_palabras_clave(consulta, corpus, top_k=3):
    if not consulta.strip():
        raise ValueError("La consulta no puede estar vacía.")
    if not 1 <= top_k <= len(corpus):
        raise ValueError(f"top_k debe estar entre 1 y {len(corpus)}.")

    tokens_consulta = tokenizar_simple(consulta)
    resultados = []

    for indice, texto in enumerate(corpus):
        coincidencias = tokens_consulta & tokenizar_simple(texto)
        resultados.append({
            "indice": indice,
            "texto": texto,
            "score": len(coincidencias),
            "coincidencias": sorted(coincidencias),
        })

    resultados.sort(key=lambda resultado: (-resultado["score"], resultado["indice"]))
    for rank, resultado in enumerate(resultados[:top_k], start=1):
        resultado["rank"] = rank

    return resultados[:top_k]

## Comparación de resultados

In [6]:
def imprimir_resultados(titulo, resultados):
    print(f"\n{titulo}")
    for resultado in resultados:
        if "coincidencias" in resultado:
            palabras = ", ".join(resultado["coincidencias"]) or "sin coincidencias"
            detalle = f"score={resultado['score']} | coincidencias={palabras}"
        else:
            detalle = f"score={resultado['score']:.4f}"
        print(
            f"  {resultado['rank']}. índice={resultado['indice']} | "
            f"{detalle} | {resultado['texto']}"
        )


def comparar_busquedas(consulta, top_k=TOP_K):
    semanticos = buscar_semanticamente(
        consulta, CORPUS, embeddings_corpus, modelo, top_k
    )
    palabras_clave = buscar_por_palabras_clave(consulta, CORPUS, top_k)

    print("=" * 100)
    print(f"CONSULTA: {consulta}")
    print("=" * 100)
    imprimir_resultados("Búsqueda semántica", semanticos)
    imprimir_resultados("Búsqueda por palabras clave", palabras_clave)

    return {"semantica": semanticos, "palabras_clave": palabras_clave}


def consultar(consulta, top_k=TOP_K):
    return comparar_busquedas(consulta, top_k)

In [7]:
resultados_pruebas = {}

for consulta in CONSULTAS:
    resultados_pruebas[consulta] = comparar_busquedas(consulta)
    print()

CONSULTA: La aplicación móvil no carga correctamente

Búsqueda semántica
  1. índice=7 | score=0.9863 | La aplicación móvil no carga correctamente.
  2. índice=8 | score=0.4524 | La aplicación se cierra sola después de abrirla.
  3. índice=2 | score=0.2873 | La cuenta fue bloqueada por demasiados intentos fallidos.

Búsqueda por palabras clave
  1. índice=7 | score=6 | coincidencias=aplicacion, carga, correctamente, la, movil, no | La aplicación móvil no carga correctamente.
  2. índice=4 | score=2 | coincidencias=la, no | La verificación en dos pasos no envía el código de seguridad.
  3. índice=8 | score=2 | coincidencias=aplicacion, la | La aplicación se cierra sola después de abrirla.

CONSULTA: Perdí mis credenciales y necesito volver a entrar

Búsqueda semántica
  1. índice=1 | score=0.5082 | Olvidé mi contraseña y necesito recuperar el acceso.
  2. índice=16 | score=0.4847 | Quiero devolver un producto porque llegó dañado.
  3. índice=0 | score=0.4257 | No puedo iniciar sesión en

## Probar una consulta propia



In [9]:
consultar("necesito hablar con una persona", top_k=3);

CONSULTA: necesito hablar con una persona

Búsqueda semántica
  1. índice=21 | score=0.6513 | Necesito comunicarme con un agente de soporte.
  2. índice=15 | score=0.3596 | Necesito modificar la dirección de entrega de mi pedido.
  3. índice=12 | score=0.3041 | El pedido ya fue enviado y está en camino.

Búsqueda por palabras clave
  1. índice=21 | score=2 | coincidencias=con, necesito | Necesito comunicarme con un agente de soporte.
  2. índice=1 | score=1 | coincidencias=necesito | Olvidé mi contraseña y necesito recuperar el acceso.
  3. índice=13 | score=1 | coincidencias=con | Puedes rastrear el paquete con el número de seguimiento.


## Reflexión requerida

Al compararse ambos modelos se ve que la busqueda semantica fue mas utili cuando la consuleta y el documento expresaban la misma idea aunque con palabras diferentes, Por ejemplo "EL pago de mi pedido aparece duplicado", recupera correctamente "Me cobraron 2 veces por la misma compra", con similitud del 0.75, y con el otro metodo de busqueda por palabras clave, funciono muy bien cuando existia una coincidencia casi exacta, donde encontro directamente la oracion correspondinete con ses palabras compartidas.

Considerando lo anterior de los metodos vistos, se podria decir que un resultado inesperado fue cuando para la consulta sobre credenciales, la busqueda semantica coloco en segunda lugar la devolucion de producto. Esto demuestra que el modelo puede confundir textos por relaciones generales entre sus palabras. En especial si el corpues no es tan grande.


Por lo que se observa las limitaciones recaen en la presencia de resultados nada relevantes, puntajes moderados y consultas ambiguas, como tipo "La cuenta no funciona" ahi no sabemos a que cuenta se refiere, por lo que para mejorar el sistema se debe ampliar el corpus con mas ejemplos por categoria, eliminar palabras vacias en la busqueda tradicional se combinara ambos metodos mediante una busqueda hibrida asi solo tener una sola y unica funcion, al igual que tener un porcentaje minimo para eliminar resultados que son solo ruido. 

